In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.utils.validation import check_is_fitted

RANDOM_STATE = 42

def load_data(data_dir):
    """
    Загружает и объединяет три CSV-файла датасета InSDN.
    Возвращает объединённый DataFrame с колонками label и attack_type.
    """
    required_files = ['Normal_data.csv', 'OVS.csv', 'metasploitable-2.csv']
    for f in required_files:
        if not os.path.exists(os.path.join(data_dir, f)):
            raise FileNotFoundError(f"Файл {f} не найден в {data_dir}")

    normal = pd.read_csv(os.path.join(data_dir, 'Normal_data.csv'))
    ovs = pd.read_csv(os.path.join(data_dir, 'OVS.csv'))
    meta = pd.read_csv(os.path.join(data_dir, 'metasploitable-2.csv'))

    # Метки: 0 — норма, 1 — атака
    normal['label'] = 0
    normal['attack_type'] = 'benign'

    ovs['label'] = 1
    meta['label'] = 1

    attacks = pd.concat([ovs, meta], ignore_index=True)

    if 'type' in attacks.columns:
        attacks['attack_type'] = attacks['type']
        attacks.drop(columns=['type'], inplace=True)
    else:
        attacks['attack_type'] = 'unknown_attack'

    df = pd.concat([normal, attacks], ignore_index=True)

    return df

def preprocess(df):
    """
    Предобработка данных: удаление нечисловых признаков, обработка пропусков,
    удаление константных признаков, масштабирование.
    """
    # Сохраняем только целевые метки
    labels = df[['label', 'attack_type']]

    # 1. Удаляем все нечисловые колонки (кроме меток)
    # Оставляем только те колонки, которые можно привести к float
    # Используем .copy(), чтобы избежать SettingWithCopyWarning
    numeric_df = df.drop(columns=['label', 'attack_type']).apply(pd.to_numeric, errors='coerce').copy()

    # Объединяем числовые данные с метками
    df_clean = pd.concat([numeric_df, labels], axis=1)

    # 2. Обработка пропусков: заполняем медианой (или 0 для полностью пустых)
    num_cols = [c for c in df_clean.columns if c not in ['label', 'attack_type']]

    for col in num_cols:
        if df_clean[col].isnull().sum() > 0:
            # Если в колонке есть хотя бы 2 уникальных значения, заполняем медианой
            if df_clean[col].nunique(dropna=True) > 1:
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            else:
                df_clean[col] = df_clean[col].fillna(0)

    # 3. Удаляем константные признаки.
    # Ищем их в ИСХОДНОМ DataFrame (df), а не в df_clean,
    # чтобы избежать KeyError для уже удаленных строковых колонок.
    const_cols = []
    for col in num_cols:
        # Проверяем только те колонки, которые еще остались в df_clean
        if col in df_clean.columns:
            if df_clean[col].nunique(dropna=True) == 1:
                const_cols.append(col)

    if const_cols:
        df_clean.drop(columns=const_cols, inplace=True)
        print(f"Удалены константные признаки: {const_cols}")

    # 4. Масштабирование числовых признаков
    scaler = StandardScaler()
    cols_to_scale = [c for c in df_clean.columns if c not in ['label', 'attack_type']]

    if cols_to_scale:
        print(f"Масштабирование признаков: {cols_to_scale}")
        df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])
    else:
        print("Нет числовых признаков для масштабирования.")

    return df_clean

def split_data(df):
    """
    Стратифицированное разделение на train/val/test по метке label.
    """
    X = df.drop(columns=['label', 'attack_type'])
    y_binary = df['label']

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y_binary, test_size=0.2, stratify=y_binary, random_state=RANDOM_STATE
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE
    )

    return (X_train, X_val, X_test), (y_train, y_val, y_test)

def train_binary_models(X_train, X_val, y_train, y_val):
    """
    Обучение и подбор гиперпараметров для бинарных моделей.
    Возвращает словарь с моделями и их результатами на валидации.
    """
    models = {
        "LogisticRegression": LogisticRegression(random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }

    param_grids = {
        "LogisticRegression": {
            "C": [0.1, 1.0],
            "penalty": ["l2"],
            "solver": ["lbfgs"]
        },
        "RandomForest": {
            "n_estimators": [100],
            "max_depth": [None, 15],
            "min_samples_split": [2]
        },
        "GradientBoosting": {
            "n_estimators": [100],
            "learning_rate": [0.1],
            "max_depth": [3]
        },
    }

    results = {}

    for name, model in models.items():
        print(f"Подбор параметров для {name}...")
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        grid = GridSearchCV(
            model,
            param_grids[name],
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1,
            verbose=0
        )
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        val_preds = best_model.predict(X_val)

        results[name] = {
            "model": best_model,
            "best_params": grid.best_params_,
            "val_metrics": {
                "accuracy": accuracy_score(y_val, val_preds),
                "precision": precision_score(y_val, val_preds),
                "recall": recall_score(y_val, val_preds),
                "f1": f1_score(y_val, val_preds),
                "roc_auc": roc_auc_score(y_val,
                                         best_model.predict_proba(X_val)[:, 1])
                                         if hasattr(best_model, "predict_proba") else None,
            }
        }

    return results

def train_multiclass_model(X_train_full, y_train_full, attack_types_train):
    """
    Обучение модели для многоклассовой классификации типов атак.
    """
    mask_attack = (y_train_full == 1)

    if mask_attack.sum() < 2:
        print("Недостаточно атакующих примеров для многоклассовой классификации.")
        return None

    X_mc_train = X_train_full[mask_attack]
    y_mc_train = attack_types_train[mask_attack]

    if len(y_mc_train.unique()) < 2:
        print("Недостаточно уникальных типов атак для многоклассовой классификации.")
        return None

    model_mc = RandomForestClassifier(random_state=RANDOM_STATE)

    param_grid_mc = {
        "n_estimators": [100],
        "max_depth": [None, 20],
        "min_samples_split": [2]
    }

    cv_mc = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    grid_mc = GridSearchCV(
        model_mc,
        param_grid_mc,
        cv=cv_mc,
        scoring="f1_weighted",
        n_jobs=-1,
        verbose=0
    )

    grid_mc.fit(X_mc_train, y_mc_train)

    return grid_mc.best_estimator_

def evaluate(models_results, multiclass_model,
             X_test_bin, y_test_bin,
             X_test_full, y_test_full,
             attack_types_test):

                 metrics_bin = {}
                 best_roc_auc = -1

                 for name, res in models_results.items():
                     model = res["model"]
                     preds = model.predict(X_test_bin)

                     metrics_bin[name] = {
                         "accuracy": accuracy_score(y_test_bin, preds),
                         "precision": precision_score(y_test_bin, preds),
                         "recall": recall_score(y_test_bin, preds),
                         "f1": f1_score(y_test_bin, preds),
                         "roc_auc": roc_auc_score(y_test_bin,
                                                  model.predict_proba(X_test_bin)[:, 1])
                                                  if hasattr(model, "predict_proba") else None,
                         "confusion_matrix": confusion_matrix(y_test_bin, preds).tolist()
                     }

                     current_roc_auc = metrics_bin[name]["roc_auc"]
                     if current_roc_auc is not None and current_roc_auc > best_roc_auc:
                         best_roc_auc = current_roc_auc

                         cm = confusion_matrix(y_test_bin, preds)

                         plt.figure(figsize=(6,5))
                         sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                                     xticklabels=['Норма', 'Атака'], yticklabels=['Норма', 'Атака'])
                         plt.xlabel('Предсказано')
                         plt.ylabel('Реально')
                         plt.title(f'Confusion Matrix: {name}')
                         plt.tight_layout()

                         try:
                             cm_path = f'cm_best_binary_{name}.png'
                             plt.savefig(cm_path)
                             plt.close()
                             print(f"Сохранена confusion matrix для лучшей бинарной модели: {cm_path}")
                             metrics_bin[name]["confusion_matrix_path"] = cm_path
                         except Exception as e:
                             plt.close()
                             print(f"Ошибка сохранения графика: {e}")

                 metrics_multi = {}
                 if multiclass_model is not None and check_is_fitted(multiclass_model):

                     mask_attack_test = (y_test_full == 1)

                     if mask_attack_test.sum() > 0:

                         X_multi_test = X_test_full[mask_attack_test]

                         preds_multi = multiclass_model.predict(X_multi_test)

                         report_multi = classification_report(
                             attack_types_test[mask_attack_test], preds_multi,
                             output_dict=True
                         )

                         metrics_multi["classification_report"] = report_multi

                         cm_multi = confusion_matrix(attack_types_test[mask_attack_test], preds_multi)

                         plt.figure(figsize=(8,6))

                         sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
                                     xticklabels=multiclass_model.classes_,
                                     yticklabels=multiclass_model.classes_)

                         plt.xlabel('Предсказано')
                         plt.ylabel('Реально')
                         plt.title('Confusion Matrix: Многоклассовая классификация')

                         plt.tight_layout()

                         try:
                             plt.savefig('cm_multiclass.png')
                             plt.close()
                             print("Сохранена confusion matrix для многоклассовой модели: cm_multiclass.png")

                             metrics_multi["confusion_matrix"] = cm_multi.tolist()

                             with open('multiclass_report.txt', 'w') as f:
                                 f.write(classification_report(
                                     attack_types_test[mask_attack_test], preds_multi))
                                 print("Сохранён отчёт по многоклассовой классификации: multiclass_report.txt")

                         except Exception as e:
                             plt.close()
                             print(f"Ошибка сохранения графика/отчёта: {e}")
                     else:
                         print("Нет атакующих примеров в тестовой выборке для многоклассовой оценки.")

                 return metrics_bin, metrics_multi

def save_results(models_results,
                 multiclass_model,
                 X_test_full,
                 y_test_full,
                 attack_types_test,
                 metrics_bin,
                 metrics_multi):
                     """
                     Сохранение предсказаний моделей и метрик.
                     """
                     predictions_df = pd.DataFrame({
                         "true_label_binary": y_test_full.values,
                         "true_attack_type": attack_types_test.values
                     })

                     for name in models_results.keys():
                         model = models_results[name]["model"]
                         preds = model.predict(X_test_full)
                         predictions_df[f"pred_{name}"] = preds

                     if multiclass_model is not None and check_is_fitted(multiclass_model):
                         mask_attack = (y_test_full == 1)
                         preds_multi = multiclass_model.predict(X_test_full[mask_attack])
                         predictions_df.loc[mask_attack, "pred_multiclass"] = preds_multi

                     predictions_df.to_csv("predictions.csv", index=False)
                     print("Сохранены предсказания в predictions.csv")

                     results_json = {
                         "binary_metrics": metrics_bin,
                         "multiclass_metrics": metrics_multi,
                         "models_info": {
                             name: {
                                 "best_params": res["best_params"],
                                 "val_metrics": res["val_metrics"]
                             } for name,res in models_results.items()
                         }
                     }

                     with open("metrics.json", "w") as f:
                         json.dump(results_json, f, indent=4)

                     print("Сохранены метрики в metrics.json")

                     joblib.dump(models_results["LogisticRegression"]["model"],
                                 "model_logreg.joblib")
                     joblib.dump(models_results["RandomForest"]["model"],
                                 "model_rf.joblib")
                     joblib.dump(models_results["GradientBoosting"]["model"],
                                 "model_gb.joblib")

                     if multiclass_model is not None:
                         joblib.dump(multiclass_model,
                                     "model_multiclass.joblib")

                     print("Сохранены обученные модели (*.joblib)")


def main(data_dir):

   """
   Основной пайплайн анализа датасета InSDN.
   """
   print("Загрузка данных...")
   df = load_data(data_dir)

   # Сводка по классам
   print("\nРаспределение классов:")
   print(df['label'].value_counts())
   print("\nРаспределение типов атак:")
   print(df['attack_type'].value_counts())

   print("\nПредобработка данных...")
   df_clean = preprocess(df)

   # Разделение на бинарные выборки (train/val/test)
   (X_train_bin, X_val_bin, X_test_bin), \
   (y_train_bin, y_val_bin, y_test_bin) = split_data(df_clean)

    # Для многоклассовой используем все данные (train+val+test признаки/метки)
   X_full = df_clean.drop(columns=['label', 'attack_type'])
   y_full = df_clean['label']
   attack_types_full = df_clean['attack_type']

     # Обучение бинарных моделей
   binary_models_res = train_binary_models(
         X_train_bin,X_val_bin,y_train_bin,y_val_bin)

    # Обучение многоклассовой модели
   multiclass_mdl = train_multiclass_model(
         pd.concat([X_train_bin,X_val_bin]),
         pd.concat([y_train_bin,y_val_bin]),
         pd.concat([df_clean.loc[y_train_bin.index,'attack_type'],
                    df_clean.loc[y_val_bin.index,'attack_type']]))

     # Оценка на тесте
   bin_metrics,multi_metrics = evaluate(
         binary_models_res,multiclass_mdl,
         X_test_bin,y_test_bin,X_full,y_full,
         attack_types_full)

   save_results(binary_models_res,multiclass_mdl,
                  X_full,y_full,
                  attack_types_full,bin_metrics,multi_metrics)

if __name__ == "__main__":
   # Для Google Colab: путь к папке с данными задаётся здесь.
   # Если данные в текущей папке:
   data_dir = "./data"
   # Если данные на Google Drive (пример):
   # from google.colab import drive
   # drive.mount('/content/drive')
   # data_dir = "/content/drive/MyDrive/InSDN"

   main(data_dir)

Загрузка данных...

Распределение классов:
label
1    275465
0     68424
Name: count, dtype: int64

Распределение типов атак:
attack_type
unknown_attack    275465
benign             68424
Name: count, dtype: int64

Предобработка данных...
Удалены константные признаки: ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Fwd PSH Flags', 'Fwd URG Flags', 'CWE Flag Count', 'ECE Flag Cnt', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Init Fwd Win Byts', 'Fwd Seg Size Min', 'Label']
Масштабирование признаков: ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT M